# Planning Agent — LangGraph (Router + Orchestrator-Worker)

Implements the workflow:

```
__start__ -> router -> [research?] -> orchestrator -> worker(s) -> reducer -> __end__
```

**How it's built:**

- **router** — looks at the incoming `task` and decides whether extra research is
  needed before planning. It routes to `research` when the task references
  facts/context the model doesn't already have, otherwise it skips straight to
  `orchestrator`.
- **research** — gathers background notes on the task (LLM call standing in for a
  real search/tool call). Always flows into `orchestrator`.
- **orchestrator** — the planner. Breaks the task into a dynamic list of
  subtask `sections`, then fans out to one `worker` per section using the
  `Send` API (this is what lets the number of workers vary per task).
- **worker** — executes a single section/subtask and writes its result back
  into a shared, `operator.add`-reduced list in state.
- **reducer** — combines every worker's output into one final report.


In [ ]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

# plain LLM for free-form generation (research notes, worker output, final report)
model = ChatGroq(model="llama-3.1-8b-instant")

# NOTE: llama-3.1-8b-instant does NOT support method="json_schema" on Groq.
# openai/gpt-oss-20b does, so it's used whenever structured output is needed
# (router's routing decision, orchestrator's plan).
structured_llm = ChatGroq(model="openai/gpt-oss-20b")

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal


class RouteDecision(BaseModel):
    needs_research: bool = Field(
        description="True if the task needs background research before it can be planned"
    )
    reason: str = Field(description="One line explanation for the decision")


class Section(BaseModel):
    name: str = Field(description="Short name of the subtask/section")
    description: str = Field(description="What this subtask/section should cover")


class Plan(BaseModel):
    sections: list[Section] = Field(
        description="Ordered list of subtasks that together accomplish the task"
    )


router_model = structured_llm.with_structured_output(
    RouteDecision, method="json_schema"
)
planner_model = structured_llm.with_structured_output(Plan, method="json_schema")

In [ ]:
from typing import TypedDict, Annotated
import operator


class PlanningState(TypedDict):
    task: str
    needs_research: bool
    research_notes: str
    sections: list[Section]
    completed_sections: Annotated[list[str], operator.add]
    final_report: str


class WorkerState(TypedDict):
    task: str
    research_notes: str
    section: Section
    completed_sections: Annotated[list[str], operator.add]

In [ ]:
def router(state: PlanningState) -> PlanningState:
    prompt = (
        "Decide whether the following task requires background research "
        "(facts, context, or up-to-date information) before it can be broken "
        f"into subtasks:\n\n{state['task']}"
    )
    decision = router_model.invoke(prompt)
    return {"needs_research": decision.needs_research}


def route_decision(state: PlanningState) -> Literal["research", "orchestrator"]:
    return "research" if state["needs_research"] else "orchestrator"

In [ ]:
def research(state: PlanningState) -> PlanningState:
    prompt = f"Provide concise background research notes useful for planning this task:\n\n{state['task']}"
    notes = model.invoke(prompt).content
    return {"research_notes": notes}

In [ ]:
from langgraph.types import Send


def orchestrator(state: PlanningState) -> PlanningState:
    prompt = (
        f"Break the following task into a list of independent subtasks/sections "
        f"that can be worked on separately:\n\nTask: {state['task']}\n\n"
        f"Research notes (if any): {state.get('research_notes', 'None')}"
    )
    plan = planner_model.invoke(prompt)
    return {"sections": plan.sections}


def assign_workers(state: PlanningState) -> list[Send]:
    # fan out — one worker per planned section, dispatched dynamically via Send
    return [
        Send(
            "worker",
            {
                "task": state["task"],
                "research_notes": state.get("research_notes", ""),
                "section": section,
            },
        )
        for section in state["sections"]
    ]

In [ ]:
def worker(state: WorkerState) -> WorkerState:
    section = state["section"]
    prompt = (
        f"Overall task: {state['task']}\n"
        f"Research notes: {state.get('research_notes', 'None')}\n\n"
        f"Complete this subtask:\n{section.name} — {section.description}"
    )
    result = model.invoke(prompt).content
    return {"completed_sections": [f"## {section.name}\n{result}"]}

In [ ]:
def reducer(state: PlanningState) -> PlanningState:
    combined = "\n\n".join(state["completed_sections"])
    return {"final_report": combined}

In [ ]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(PlanningState)

graph.add_node("router", router)
graph.add_node("research", research)
graph.add_node("orchestrator", orchestrator)
graph.add_node("worker", worker)
graph.add_node("reducer", reducer)

graph.add_edge(START, "router")
graph.add_conditional_edges(
    "router", route_decision, {"research": "research", "orchestrator": "orchestrator"}
)
graph.add_edge("research", "orchestrator")
graph.add_conditional_edges("orchestrator", assign_workers, ["worker"])
graph.add_edge("worker", "reducer")
graph.add_edge("reducer", END)

workflow = graph.compile()

In [ ]:
from IPython.display import Image, display

display(Image(workflow.get_graph().draw_mermaid_png()))

In [ ]:
final_state = workflow.invoke(
    {"task": "Plan a 3-day itinerary for a first-time visitor to Pune, India"}
)

print("Needed research:", final_state["needs_research"])
print("\nSections planned:", [s.name for s in final_state["sections"]])
print("\nFinal report:\n")
print(final_state["final_report"])